In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    return response.output_text

In [4]:
question = 'I just discovered the course. Can I join the course now?'
answer = llm('question')
print(answer)

Sure — what’s your question?


In [5]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [6]:

prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [7]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join the course now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Stude

In [8]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [9]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 144},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253}]

In [11]:
documents = []

url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [12]:
documents[700]

{'id': 'afd915ae5b',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 4. Evaluation Metrics for Classification',
 'question': 'What exactly is taught in the evaluation module?',
 'answer': 'In Module 4, you’ll learn the core evaluation concepts used in classification tasks. Topics include:\n- Metrics and diagnostics: precision, recall, ROC curves, and precision-recall curves\n- Evaluation mindsets: how to think critically about metrics and validation in ML projects\n- Common pitfalls: data leakage, improper validation, misinterpretation of metrics and curves\n- Practical interpretation: selecting metrics based on context (class balance, costs of errors) and conveying results clearly\n- Real-world applicability: how these concepts guide model comparison, threshold selection, and reporting\nThis module is designed to be conceptual and abstract, yet practical for real-world ML work.'}

In [13]:
from minsearch import Index

index = Index(
    text_fields = ['question', 'section', 'answer'],
    keyword_fields = ['course']
)

index.fit(documents)

In [14]:
search_results = index.search(
    question, 
    boost_dict = {'question':2.0, 'answer':1.0, 'section':0.5},
    filter_dict = {"course":"llm-zoomcamp"},
    num_results = 5
    )

In [15]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2.0, 'answer':1.0, 'section':0.5}
    filter_dict = {'course':course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results=5
    )

In [16]:
search_results = search(question)

In [17]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge i

## Building Prompt

In [18]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [19]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [20]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [21]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalk

In [22]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

'\nQuestion:\nI just discovered the course. Can I join the course now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: How should I start the course and follow the weekly workflow?\nA: Start with the [LLM Zoomcamp docs](https:

In [23]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
        )
    return prompt.strip()


In [24]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join the course now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/c

## LLM Section

In [25]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [26]:
response.output_text

'Yes — you can join now.\n\nYou can start the course at any time and work through the materials. If you want a certificate, make sure to submit your project while the course is still accepting submissions.'

In [27]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0c04138444c9cdb6006a7b9857834c8194ba9a7503afce332c",
  "created_at": 1786484823.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0c04138444c9cdb6006a7b9858170081948f5bb5ac29fa07b3",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can join now.\n\nYou can start the course at any time and work through the materials. If you want a certificate, make sure to submit your project while the course is still accepting submissions.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1786484824.0,
  "conversat

In [28]:
response.output[0].content[0].text

'Yes — you can join now.\n\nYou can start the course at any time and work through the materials. If you want a certificate, make sure to submit your project while the course is still accepting submissions.'

In [29]:
response.usage

ResponseUsage(input_tokens=496, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=45, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=541)

In [30]:
input_price = 0.75/1_000_000
output_price = 4.50/1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

print(cost)

0.0005745


In [31]:
message_hisstory = [
    {'role': 'system', 'content': INSTRUCTIONS}, # System prompt
    {'role': 'user', 'content': prompt} # User prompt, changeable 
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [32]:
response.output_text

'Yes — you can join now.\n\nYou can start learning anytime, but if you want a certificate, you need to submit your project while submissions are still open.'

In [ ]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'system', 'content': INSTRUCTIONS}, # System prompt
        {'role': 'user', 'content': prompt} # User prompt, changeable 
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [34]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model = model)

    return answer

In [35]:
answer = rag(question)
print(answer)

Yes, you can still join the course now. If you want a certificate, you’ll need to submit your project while submissions are still being accepted.
